# Features on a Base Map

This tutorial reads a file of polygons stored in a projected coordinate system, moves them into longitude and latitude with DuckDB and its spatial extension, and shows them on an OpenStreetMap base map with `Pollis.webmap`: first in one colour, then coloured by a numeric column and by a text column, and with a second layer of points. The base maps come from OpenFreeMap's vector tiles and are drawn by MapLibre GL JS.

In [ ]:
using DuckDB, Tables

con = DBInterface.connect(DuckDB.DB)  # an in-memory database
DBInterface.execute(con, "LOAD spatial")
DBInterface.execute(con, "SET geometry_always_xy = true")  # longitude, latitude in every transform

## 1. A File in a Projected Coordinate System

To keep the tutorial self-contained, we first write a small GeoPackage in a temporary folder: the areas within 20 km of six European cities, with their distance from Amsterdam in km. The file is stored in ETRS89 / LAEA Europe (`EPSG:3035`), a projected system in metres, as European statistical agencies often publish their data. Replace `path` with your own file and `EPSG:3035` with its coordinate system.

In [ ]:
dir = mktempdir()
path = joinpath(dir, "areas.gpkg")
cities = "SELECT name, ST_Transform(ST_Point(lon, lat), 'EPSG:4326', 'EPSG:3035') AS geom FROM (VALUES ('Rome', 12.4964, 41.9028), ('Paris', 2.3522, 48.8566), ('London', -0.1276, 51.5072), ('Amsterdam', 4.9041, 52.3676), ('Munich', 11.582, 48.1351), ('Prague', 14.4378, 50.0755)) t(name, lon, lat)"
amsterdam = "ST_Transform(ST_Point(4.9041, 52.3676), 'EPSG:4326', 'EPSG:3035')"
areas = "SELECT name, CASE WHEN name IN ('Rome', 'Munich', 'Prague') THEN 'east' ELSE 'west' END AS region, round(ST_Distance(geom, $amsterdam) / 1000, 1) AS km_to_amsterdam, ST_Buffer(geom, 20000) AS geom FROM ($cities)"
DBInterface.execute(con, "COPY ($areas) TO '$(path)' (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'areas', SRS 'EPSG:3035')")

## 2. Read the File and Look at Its Coordinates

`ST_Read` reads the GeoPackage through GDAL. Its coordinates are metres east and north on the LAEA Europe grid, not longitude and latitude: a web map given them would put the features far outside the world.

In [ ]:
DBInterface.execute(con, "CREATE OR REPLACE TABLE raw AS SELECT * FROM ST_Read('$(path)')")
box = Tables.columntable(DBInterface.execute(con, "SELECT min(ST_XMin(geom)) AS xmin, max(ST_XMax(geom)) AS xmax, min(ST_YMin(geom)) AS ymin, max(ST_YMax(geom)) AS ymax FROM raw"))
println("x from ", round(box.xmin[1]), " to ", round(box.xmax[1]), ", y from ", round(box.ymin[1]), " to ", round(box.ymax[1]))

## 3. Move the Features Into Longitude and Latitude

Web maps take longitude and latitude on WGS 84 (`EPSG:4326`), as GeoJSON does. `ST_Transform` moves the features there from `EPSG:3035`; GDAL's own feature number, `OGC_FID` in some formats, is dropped if present. We then check that every coordinate is within longitude -180 to 180 and latitude -90 to 90.

In [ ]:
DBInterface.execute(con, "CREATE OR REPLACE TABLE features AS SELECT * EXCLUDE (geom), ST_Transform(geom, 'EPSG:3035', 'EPSG:4326') AS geom FROM raw")
DBInterface.execute(con, "ALTER TABLE features DROP COLUMN IF EXISTS ogc_fid")
r = Tables.columntable(DBInterface.execute(con, "SELECT count(*) AS n, count(*) FILTER (WHERE ST_XMin(geom) < -180 OR ST_XMax(geom) > 180 OR ST_YMin(geom) < -90 OR ST_YMax(geom) > 90) AS outside, min(ST_XMin(geom)) AS xmin, max(ST_XMax(geom)) AS xmax FROM features"))
println(r.n[1], " features, ", r.outside[1], " outside longitude and latitude; longitude from ", round(r.xmin[1], digits = 3), " to ", round(r.xmax[1], digits = 3))

## 4. Show the Features on a Base Map

The table is written as a GeoJSON layer, which `Pollis.layer` names and `Pollis.webmap` shows on a base map. Without a centre and a zoom, the map fits the features. Click an area to see its attributes.

In [ ]:
layer = joinpath(dir, "areas.geojson")
DBInterface.execute(con, "COPY features TO '$(layer)' (FORMAT GDAL, DRIVER 'GeoJSON')")
Pollis.webmap(Pollis.layer(layer; name = "Areas"); style = "positron")

## 5. Colour the Features by a Numeric Column

With `column`, each feature is coloured by the value of that column. For a numeric column, the colours run on a scale from the smallest value, dark purple, to the largest, yellow, and the legend shows both ends. Positron, light grey, keeps the scale readable.

In [ ]:
Pollis.webmap(Pollis.layer(layer; name = "Distance from Amsterdam", column = "km_to_amsterdam", opacity = 0.7); style = "positron", title = "Areas within 20 km of six cities")

## 6. Colour the Features by a Text Column

A text column is coloured by category: one colour for each of its ten most common values, listed in the legend. Here the style is Bright, which shows roads, parks and water in colour.

In [ ]:
Pollis.webmap(Pollis.layer(layer; name = "Region", column = "region"); style = "bright")

## 7. Two Layers on One Map

A map can hold several layers, drawn in the order given, the first at the bottom. Here the cities themselves, as points at the centres of the areas, go on top of the areas, in a colour of our own.

In [ ]:
points = joinpath(dir, "cities.geojson")
DBInterface.execute(con, "COPY (SELECT name, ST_Centroid(geom) AS geom FROM features) TO '$(points)' (FORMAT GDAL, DRIVER 'GeoJSON')")
Pollis.webmap(Pollis.layer(layer; name = "Areas", color = "#9e9e9e", opacity = 0.3), Pollis.layer(points; name = "Cities", color = "#e15759"); style = "positron")

## 8. A Place at a Given Zoom

With a centre, longitude first, and a zoom, the map opens on that place rather than fitting the layers: zoom 12 shows a city, 15 a neighbourhood. Here, Amsterdam at zoom 9, with the whole of its area around it.

In [ ]:
Pollis.webmap(Pollis.layer(layer; name = "Areas", opacity = 0.2); style = "liberty", center = (4.9041, 52.3676), zoom = 9)